# Phase 10 — Final Cross-Version Evaluation

This notebook is the interactive interface to the measured Phase 10 comparison. It delegates to the tested final-evaluation runner, compares the eight PRD versions, and displays Recall@5, strict grounded faithfulness, latency, target checks, and question-level trade-offs.

It implements no experiment dashboard, Streamlit UI, or later roadmap work.

## Setup

Use the repository `.venv` kernel. Running the experiment requires the Phase 2–8 result artifacts, Ollama, Pinecone, and the configured local models.

In [ ]:
from __future__ import annotations

import json
import shlex
import subprocess
import sys
from pathlib import Path

import yaml
from IPython.display import JSON, Markdown, display

PROJECT_ROOT = next(
    (
        candidate.resolve()
        for candidate in (Path.cwd(), Path.cwd().parent)
        if (candidate / "pyproject.toml").is_file()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from the repository root or notebooks directory.")


def run_command(command: list[object]) -> None:
    normalized = [str(part) for part in command]
    print(shlex.join(normalized))
    subprocess.run(normalized, cwd=PROJECT_ROOT, check=True)


def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))


print(f"Project root: {PROJECT_ROOT}")
print(f"Kernel Python: {sys.executable}")

## Configuration

Review the exact eight-version matrix. External execution remains opt-in because the runner rebuilds only its guarded `phase10-*` namespace and evaluates 30 live runtime questions.

In [ ]:
CONFIG_PATH = PROJECT_ROOT / "config" / "final_evaluation.yaml"
QUESTIONS_PATH = PROJECT_ROOT / "evaluation" / "questions.json"
OUTPUT_ROOT = PROJECT_ROOT / "evaluation" / "results" / "phase10_final"
RUN_EXPERIMENT = False  # Set to True, then run the execution cell.

configuration = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
display(JSON(configuration))

## Run the final evaluation

In [ ]:
COMMAND = [
    sys.executable,
    PROJECT_ROOT / "evaluation" / "run_final_evaluation.py",
    "--config", CONFIG_PATH,
    "--questions", QUESTIONS_PATH,
    "--output-root", OUTPUT_ROOT,
]

if RUN_EXPERIMENT:
    run_command(COMMAND)
else:
    print("Dry run. Set RUN_EXPERIMENT = True to execute:")
    print(shlex.join(str(part) for part in COMMAND))

## Inspect the measured table and recommendation

In [ ]:
COMPARISON_PATH = OUTPUT_ROOT / "comparison.json"

if COMPARISON_PATH.exists():
    comparison = read_json(COMPARISON_PATH)
    table = [
        {
            "version": version["label"],
            "recall_at_5": version["metrics"]["recall_at_5"],
            "faithfulness": version["metrics"]["faithfulness"],
            "average_latency_seconds": version["metrics"]["average_latency_seconds"],
            "p95_latency_seconds": version["metrics"]["p95_latency_seconds"],
        }
        for version in comparison["versions"]
    ]
    display(JSON({
        "versions": table,
        "recommendation": comparison["recommendation"],
        "success_targets": comparison["success_targets"],
    }))
else:
    print(f"No Phase 10 comparison artifact yet: {COMPARISON_PATH}")

## Inspect why each version changed

In [ ]:
ANALYSIS_PATH = OUTPUT_ROOT / "analysis.md"

if ANALYSIS_PATH.exists():
    display(Markdown(ANALYSIS_PATH.read_text(encoding="utf-8")))
else:
    print(f"No Phase 10 analysis artifact yet: {ANALYSIS_PATH}")